In [1]:
import numpy as np
import pandas as pd
import MetaTrader5 as mt5
import time
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
import ta

In [2]:
def features_engineering(df):
    """ This function which creates all the necessary sets for the algorithms"""

    # Allows the variables to be call outside the function
    global X_train
    global X_test
    global y_train_reg
    global y_train_cla 
    global X_train_scaled 
    global X_test_scaled
    global split_train_test
    global split_test_valid
    global X_valid
    global X_valid_scaled
    global X_train_pca
    global X_test_pca
    global X_val_pca


    # Create ours own metrics to compute the strategy returns
    df["returns"] = ((df["close"] - df["close"].shift(1)) / df["close"])
    df["sLow"] = ((df["low"] - df["close"].shift(1)) / df["close"].shift(1))
    df["sHigh"] = ((df["high"] - df["close"].shift(1)) / df["close"].shift(1))

    # Features engineering
    df["returns t-1"] = df[["returns"]].shift(1)

    # Mean of returns
    df["mean returns 15"] = df[["returns"]].rolling(15).mean().shift(1)
    df["mean returns 60"] = df[["returns"]].rolling(60).mean().shift(1)

    # Volatility of returns
    df["volatility returns 15"] = df[["returns"]].rolling(15).std().shift(1)
    df["volatility returns 60"] = df[["returns"]].rolling(60).std().shift(1)

    # Drop missing values
    df = df.dropna()
    
    # Percentage train set
    split = int(0.80*len(df))
   

   
    list_x = ["returns t-1", "mean returns 15", "mean returns 60",
                  "volatility returns 15",
                  "volatility returns 60"]


    split_train_test = int(0.70*len(df))
    split_test_valid = int(0.90*len(df))

    # Train set creation
    X_train = df[list_x].iloc[:split_train_test]

    y_train_reg = df[["returns"]].iloc[:split_train_test]

    y_train_cla = np.round(df[["returns"]].iloc[:split_train_test]+0.5)


    # Test set creation
    X_test = df[list_x].iloc[split_train_test:split_test_valid]
    
    # Test set creation
    X_val = df[list_x].iloc[split_test_valid:]


    # NORMALIZATION 
    # Import the class
    from sklearn.preprocessing import StandardScaler

    # Initialize the class
    sc = StandardScaler()

    # Standardize the data
    X_train_scaled = sc.fit_transform(X_train)
    X_test_scaled = sc.transform(X_test)
    X_val_scaled = sc.transform(X_val)
    
    
    
    
    
    # PCA
    # Import the class
    from sklearn.decomposition import PCA
    
    # Initiliaze the class
    pca = PCA(n_components=3)
    
    # Apply the PCA
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)
    X_val_pca = pca.transform(X_val_scaled)
   

In [3]:
def calculate_dynamic_lot(symbol, sl_pct, risk_pct=1.0):
    """
    Calculate lot size based on account equity and permitted risk.
    
    Args:
      symbol   (str):  e.g. "XAUUSDm"
      sl_pct   (float):  Stop-loss as a fraction of price (e.g. 4.7/100)
      risk_pct (float):  % of equity you’re willing to risk per trade (default = 1%)
    
    Returns:
      float: the volume you can trade (rounded to the broker’s step-size)
    """
    # 1) get account info
    acct = mt5.account_info()
    if acct is None:
        print("❌ Could not fetch account info")
        return SYMBOL_LOTS.get(symbol, 0.01)

    equity = acct.equity
    risk_amount = equity * (risk_pct / 100.0)

    # 2) get symbol specs
    info = mt5.symbol_info(symbol)
    if info is None:
        print(f"❌ Could not fetch symbol info for {symbol}")
        return SYMBOL_LOTS.get(symbol, 0.01)

    # 3) current price
    tick = mt5.symbol_info_tick(symbol)
    price = tick.ask if tick.ask > 0 else tick.bid

    # 4) absolute SL distance in price units
    sl_dist = price * sl_pct
    if sl_dist <= 0:
        print("⚠️ SL distance zero or negative → using default lot")
        return SYMBOL_LOTS.get(symbol, 0.01)

    # 5) risk per lot in account currency:
    #    risk_per_lot = contract_size * price_move
    risk_per_lot = info.trade_contract_size * sl_dist

    # 6) compute raw lot
    raw_lot = risk_amount / risk_per_lot

    # 7) clamp and round to step
    min_vol  = info.volume_min
    max_vol  = info.volume_max
    step     = info.volume_step

    # floor to nearest step
    steps = int(raw_lot / step)
    lot   = max(min_vol, min(steps * step, max_vol))

    print(
        f"{symbol}  ⇒  Equity={equity:.2f},  RiskAmt={risk_amount:.2f},\n"
        f"    Price={price:.3f},  SLdist={sl_dist:.3f},\n"
        f"    ContractSize={info.trade_contract_size},\n"
        f"    RawLot={raw_lot:.4f} → FinalLot={lot:.2f}\n"
        f"    (min={min_vol}, max={max_vol}, step={step})"
    )

    return round(lot, 2)


In [ ]:

import time
from datetime import datetime, timezone
from joblib import load

# ── USER CONFIG ────────────────────────────────────────────────────────────────
SYMBOLS    = ["BTCUSDm","XAUUSDm"]
MODELS_DIR = "ModelsProd"
TIMEFRAME  = mt5.TIMEFRAME_H4
TP_PCT     = 0.2 / 100    # 0.2%
SL_PCT     = 4.7 / 100    # 4.7%
MAGIC      = 49494949
DEVIATION  = 20
# ────────────────────────────────────────────────────────────────────────────────

def initialize_mt5():
    if not mt5.initialize():
        raise RuntimeError(f"MT5 init failed: {mt5.last_error()}")

def load_model(symbol):
    return load(f"{MODELS_DIR}/{symbol}_voting.joblib")

def get_latest_bar_df(symbol, n=1000, timeframe=TIMEFRAME):
    """Fetch last n bars into a DataFrame; index=datetime, columns=open,high,low,close,volume"""
    rates = mt5.copy_rates_from(symbol, timeframe, datetime.now(timezone.utc), n)
    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s')
    return df.set_index('time')

def resume(symbol):
    """Return all open positions for this symbol."""
    pos = mt5.positions_get(symbol=symbol) or []
    df = pd.DataFrame([
        {
            "ticket": p.ticket,
            "type":   p.type,    # 0=buy, 1=sell
            "symbol": p.symbol,
            "volume": p.volume,
            "profit": p.profit
        } for p in pos
    ])
    return df

def live_trade(symbol, model):
    # → 0) guard: if there are any open positions for this symbol, skip
    open_df = resume(symbol)
    if not open_df.empty:
        print(f"[{symbol}] {len(open_df)} open position(s), waiting for them to close.")
        return

    # → 1) get data & model signal
    df = get_latest_bar_df(symbol, n=500)
    features_engineering(df)   # MUST populate X_train_pca, X_test_pca, X_val_pca

    X_full    = np.concatenate((X_train_pca, X_test_pca, X_val_pca), axis=0)
    last_feat = X_full[-1].reshape(1, -1)
    pred      = model.predict(last_feat)[0]
    signal    = 1 if pred > 0 else -1

    # → 2) build order params
    tick       = mt5.symbol_info_tick(symbol)
    price      = tick.ask if signal>0 else tick.bid
    sl         = price * (1 - SL_PCT) if signal>0 else price * (1 + SL_PCT)
    tp         = price * (1 + TP_PCT) if signal>0 else price * (1 - TP_PCT)
    order_type = mt5.ORDER_TYPE_BUY if signal>0 else mt5.ORDER_TYPE_SELL
    lot        = calculate_dynamic_lot(symbol, SL_PCT, risk_pct=2.0)

    req = {
        "action":      mt5.TRADE_ACTION_DEAL,
        "symbol":      symbol,
        "volume":      lot,
        "type":        order_type,
        "price":       price,
        "sl":          sl,
        "tp":          tp,
        "deviation":   DEVIATION,
        "magic":       MAGIC,
        "comment":     "AI-CapitainVigs",
        "type_time":   mt5.ORDER_TIME_GTC,
        "type_filling":mt5.ORDER_FILLING_IOC,
    }

    res = mt5.order_send(req)
    if res.retcode != mt5.TRADE_RETCODE_DONE:
        print(f"[{symbol}] order failed → {res.comment}")
    else:
        side = "BUY" if signal>0 else "SELL"
        print(f"[{symbol}] {side} @ {price:.5f}  lot={lot}  SL={SL_PCT*100:.2f}%  TP={TP_PCT*100:.2f}%")

def main_loop():
    initialize_mt5()
    models = {s: load_model(s) for s in SYMBOLS}

    print("Starting live-trading loop. Ctrl+C to stop.")
    try:
        while True:
            now = datetime.now()
            # every H4 bar close: minute % 240 == 0
            # trade every 15 min
            if now.minute % 15 == 0 and now.second < 5:
                for sym, m in models.items():
                    live_trade(sym, m)
                time.sleep(60)
            time.sleep(1)
    except KeyboardInterrupt:
        print("Stopping.")
    finally:
        mt5.shutdown()

if __name__ == "__main__":
    main_loop()


Starting live-trading loop. Ctrl+C to stop.
